In [ ]:
from langgraph.graph import START,END,StateGraph
from langchain_ollama import ChatOllama
from typing import Literal,TypedDict
from pydantic import BaseModel,Field


In [ ]:
model=ChatOllama(model="llama3", temperature=0)

In [ ]:
class SentimentBaseModel(BaseModel):
    sentiment : Literal["Positive","Negative"]=Field(description="Sentiment of the review")
    

In [ ]:
class DiagnosisSchema(BaseModel):
    issueType:Literal["UX","Performance","Bug","Support","Other"]=Field(description="The Category of the issue mentioned in the review")
    tone:Literal["Angry","Frustrated","Disappointed","Calm"]=Field(description="The Emotional tone expressed by the user")
    urgency:Literal["Low","Medium","High"]=Field(description="Priority of the issue")
    
    
    

In [ ]:
structuredModel=model.with_structured_output(SentimentBaseModel)

structuredModel2=model.with_structured_output(DiagnosisSchema)




In [ ]:
prompt="What is the sentiment of the following review-The movie was really bad"

structuredModel.invoke(prompt).sentiment

In [ ]:
class ReviewState(TypedDict):
    review:str
    sentiment:Literal["Positive","Negative"]
    diagnosis:dict
    response:str
    

In [ ]:
def findSentiment(state : ReviewState):
    prompt=f"For the following review find out the sentiment \n {state["review"]}"
    sentiment=structuredModel.invoke(prompt).sentiment
    return {"sentiment":sentiment}


def checkSentiment(state:ReviewState)->Literal["positiveResponse", "Run_Diagnosis"]:
    if state["sentiment"]=="Positive":
        return "positiveResponse"
    
    else:
        return "Run_Diagnosis"
    
    
def positiveResponse(state:ReviewState):
    prompt=f"""Write a warm thank you message in response to this review:
    \n\n "{state['review']} "\n Also ask user to leave a review on website"""
    
    response=model.invoke(prompt).content
    
    return {"response":response}


def Run_Diagnosis(state:ReviewState):
    prompt=f"""Diagnose this negative review :\n\n {state['review']}\n
    "Return issueType,tone,urgency"""
    
    response=structuredModel2.invoke(prompt)
    
    return {'diagnosis' : response.model_dump()}



def NegativeResponse(state:ReviewState):
    diagnosis=state["diagnosis"]
    prompt=f"""You are a helpful assistant The user had a 
        '{diagnosis['issueType']}' issue, sounded '{diagnosis['tone']}',
        and marked urgency as '{diagnosis['urgency']}'.
        Write an empathetic, helpful resolution message.
        """
    response = model.invoke(prompt).content

    return {'response': response}
       

In [ ]:
graph=StateGraph(ReviewState)

graph.add_node("findSentiment",findSentiment)
graph.add_node("positiveResponse",positiveResponse)
graph.add_node("Run_Diagnosis",Run_Diagnosis)
graph.add_node("NegativeResponse",NegativeResponse)


graph.add_edge(START,"findSentiment")
graph.add_conditional_edges("findSentiment",checkSentiment)

graph.add_edge("positiveResponse",END)
graph.add_edge("Run_Diagnosis","NegativeResponse")
graph.add_edge("NegativeResponse",END)

workflow=graph.compile()


In [ ]:
workflow

In [ ]:
intial_state={
    'review': "I’ve been trying to log in for over an hour now,and the app keeps freezing on the authentication screen. I even tried reinstalling it, but no luck. This kind of bug is unacceptable, especially when it affects basic functionality."
}
workflow.invoke(intial_state)